In [31]:
from aiida.engine import run_get_node
from topworkchain import PrototypeTopWorkChain
from aiida.orm import FolderData
from aiida import load_profile
load_profile()

Profile<uuid='8982fbed0baf4361bcf833f5b70e51c4' name='presto'>

In [32]:
#Run the model
results, node = run_get_node(PrototypeTopWorkChain)
#Load the FolderData node containing the compressed cube files (a folder of files stored on the AiiDA database.)
cube_folder = results["cube_folder"]

uuid: 69b265a9-442d-4fad-995a-63fea9a78282 (pk: 416)


In [33]:
import nglview as nv
import ipywidgets as widgets
import traitlets as tl
import ase.io
import io
import os
from ase.build import molecule
import tempfile
from pathlib import Path
from ipywidgets import Layout

In [40]:
# Create an output directory for the temporary cube files
output_dir = Path("extracted_cubes")
output_dir.mkdir(exist_ok=True)

# Create a list of the cube files in the data node
names = cube_folder.list_object_names()

# Read in the cube files and write them to temporary files
for name in names:
    with cube_folder.open(name, mode="rb") as file_in:
        with open(f"{output_dir}/{name}.cube", "wb") as file_out:
            file_out.write(file_in.read())

cube_names = sorted([name for name in os.listdir("./extracted_cubes") if name.endswith(".cube")])

# Create a list of the paths to the temporary cube files
CUBE_PATHS = [f"{output_dir}/{name}.cube" for name in names]

# Create dropdown widgets for selecting the cube files
cube_file_1 = widgets.Dropdown(
    options=cube_names,
    value=cube_names[0] if cube_names else None,
    description="Molecule 1"
 )

cube_file_2 = widgets.Dropdown(
    options=cube_names,
    value=cube_names[1] if len(cube_names) > 1 else (cube_names[0] if cube_names else None),
    description="Molecule 2"
 )

# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./extracted_cubes/{cube_file_1.value}"
CUBE_PATH_2 = f"./extracted_cubes/{cube_file_2.value}"

# Create NGL views for the selected cube files
molecule1 = nv.show_ase(ase.io.read(CUBE_PATH_1))
molecule2 = nv.show_ase(ase.io.read(CUBE_PATH_2))
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube")
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube")


# Create sliders for adjusting the isovalue (not visible in the NGL view, but used to set the isovalue of the surfaces)
positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
 )
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
 )

# Create a dropdown for selecting the colour scheme of the surfaces
colour_scheme = widgets.Dropdown(
    options=["default", "electrostatic", "element"],
    value="default",
    description="Colour scheme",
 )

def get_surface_colors(scheme):
    if scheme == "default":
        return "#3366ff", "#ff4d4d"
    if scheme == "electrostatic":
        return "#f713d5", "#3366ff"
    if scheme == "element":
        return "#7a4dff", "#eeff00"
    return "#3366ff", "#ff4d4d"

# Set the initial surface colors based on the default colour scheme
positive_color, negative_color = get_surface_colors(colour_scheme.value)
status = widgets.HTML()

# Define functions to redraw the surfaces when the sliders or colour scheme are changed
def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=positive_color,
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=negative_color,
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=positive_color,
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=negative_color,
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    global positive_color, negative_color
    positive_color, negative_color = get_surface_colors(colour_scheme.value)
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(f"./extracted_cubes/{cube_file_1.value}", ext="cube")  # Add path!
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(f"./extracted_cubes/{cube_file_2.value}", ext="cube")  # Add path!
    redraw_NTO_2()

# Set up observers to redraw the surfaces when the sliders or colour scheme are changed
for control in [positive_level, negative_level, colour_scheme]:
    control.observe(redraw_surfaces, names="value")

cube_file_1.observe(update_NTO_1, names="value")
cube_file_2.observe(update_NTO_2, names="value")

redraw_surfaces()

# Arrange the widgets in a layout
controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        widgets.VBox([colour_scheme], layout = Layout(border="1px solid black", width="40%")),
        status,
    ],
 )
molecule1_box = widgets.VBox([cube_file_1, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([cube_file_2, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))
